# 4.2 – Producteur de flux – socket Python 1h – Présentiel

### Écrire un script Python simulant un flux de données taxi en émettant les lignes d'un CSV sur une socket TCP, sans charger le fichier en mémoire.



### 1. Lire le CSV ligne par ligne via un itérateur : with open("taxi.csv","r") as f: for line in f: ...


In [ ]:
with open("yellow_tripdata_2026-01.csv", "r") as f:
    for line in f:  # itérateur
        print(line)

### 2. Créer une socket TCP : socket.AF_INET, SOCK_STREAM, bind(("localhost", 9999)), listen, accept


In [46]:
import socket

# 1. Créer le socket
server = socket.socket(socket.AF_INET, socket.SOCK_STREAM)

# 2. Évite l'erreur "Address already in use" si on relance le script
server.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)

# 3. Attacher le socket au port 9999
server.bind(("localhost", 9999))

# 4. Mettre en écoute (max 1 client en attente)
server.listen(1)

print("En attente d'un client sur le port 9999...")

# 5. Accepter la connexion du client
client, address = server.accept()

print(f"Client connecté depuis {address}")

# teste
# Terminal 1 — lancer le serveur
#python3 producer.py

# Terminal 2 — se connecter
#nc localhost 9999

En attente d'un client sur le port 9999...
Client connecté depuis ('127.0.0.1', 45110)


### 3. Émettre chaque ligne encodée à 100 lignes/s : client.send((line+"\n").encode()); time.sleep(1/100)

In [47]:
import time

server = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
server.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
server.bind(("localhost", 9999))
server.listen(1)

print("En attente d'un client sur le port 9999...")
client, address = server.accept()
print(f"Client connecté depuis {address}")

with open("yellow_tripdata_2026-01.csv", "r") as f:
    for line in f:
        client.send((line + "\n").encode())  # envoie la ligne encodée
        time.sleep(1/100)                    # pause 10ms = 100 lignes/s

En attente d'un client sur le port 9999...
Client connecté depuis ('127.0.0.1', 47598)


BrokenPipeError: [Errno 32] Broken pipe

### 4. Tester la réception avec nc localhost 9999 dans un autre terminal

![emission_via_socket_100lignes_s.png](/home/youssef.hirchaou@Digital-Grenoble.local/campus/Big_data_architecture_distribuées/Streaming_Tabulaire/emission_via_socket_100lignes_s.png)

### 5. Vérifier que la RAM reste stable quel que soit le volume du fichier (htop)

In [48]:
server = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
server.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
server.bind(("localhost", 9999))
server.listen(1)

print("En attente d'un client sur le port 9999...")
client, address = server.accept()
print(f"Client connecté depuis {address}")
try:
    with open("yellow_tripdata_2026-01.csv", "r") as f:
        for i, line in enumerate(f):
            client.send((line + "\n").encode())
            time.sleep(1/100)

            # Affiche la progression toutes les 1000 lignes
            if i % 1000 == 0:
                print(f"{i} lignes envoyées...")

except BrokenPipeError:
    print("Client déconnecté.")
finally:
    client.close()
    server.close()
    print("Socket fermé proprement.")

En attente d'un client sur le port 9999...
Client connecté depuis ('127.0.0.1', 48840)
0 lignes envoyées...
1000 lignes envoyées...
2000 lignes envoyées...
Client déconnecté.
Socket fermé proprement.


## 4.3 – Job Spark Structured Streaming + fenêtres glissantes

### Connexion à la source et parsing
### 6. Créer le DataFrame de streaming :
### spark.readStream.format("socket").option("host","localhost").option("port",9999).load()

In [50]:
from pyspark.sql import SparkSession

# Créer la session Spark
spark = SparkSession.builder \
    .appName("TaxiStreaming") \
    .getOrCreate()

# Réduire les logs pour y voir plus clair
spark.sparkContext.setLogLevel("WARN")

# Lire le flux depuis le socket
raw_df = spark.readStream \
    .format("socket") \
    .option("host", "localhost") \
    .option("port", 9999) \
    .load()

# Vérification — doit afficher True
print("isStreaming :", raw_df.isStreaming)

ConnectionRefusedError: [Errno 111] Connection refused

In [51]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_csv, col, window, count, avg, to_timestamp, expr


spark = SparkSession.builder \
    .appName("TaxiStreaming") \
    .getOrCreate()
spark.sparkContext.setLogLevel("WARN")

# Schéma en string (obligatoire pour from_csv)
schema_str = """
    idx LONG,
    VendorID LONG,
    tpep_pickup_datetime STRING,
    tpep_dropoff_datetime STRING,
    passenger_count DOUBLE,
    trip_distance DOUBLE,
    RatecodeID DOUBLE,
    store_and_fwd_flag STRING,
    PULocationID LONG,
    DOLocationID LONG,
    payment_type LONG,
    fare_amount DOUBLE,
    extra DOUBLE,
    mta_tax DOUBLE,
    tip_amount DOUBLE,
    tolls_amount DOUBLE,
    improvement_surcharge DOUBLE,
    total_amount DOUBLE,
    congestion_surcharge DOUBLE,
    Airport_fee DOUBLE,
    cbd_congestion_fee DOUBLE
"""

# Connexion au socket
raw_df = spark.readStream \
    .format("socket") \
    .option("host", "localhost") \
    .option("port", 9999) \
    .load()


ConnectionRefusedError: [Errno 111] Connection refused

### 7. Parser les lignes CSV avec un StructType explicite et from_csv()

In [52]:
# Parser les lignes CSV
parsed_df = raw_df.select(
    from_csv(col("value"), schema_str).alias("data")
).select("data.*") \
 .withColumn("tpep_pickup_datetime", 
             expr("try_cast(tpep_pickup_datetime as timestamp)")) \
 .filter(col("tpep_pickup_datetime").isNotNull())  # ignore les lignes mal formées

ConnectionRefusedError: [Errno 111] Connection refused

###  8. Vérifier : df.isStreaming doit retourner True

In [30]:
print("isStreaming :", parsed_df.isStreaming)

ConnectionRefusedError: [Errno 111] Connection refused

## Agrégations sur fenêtres glissantes
### 9. Appliquer le watermarking : df.withWatermark("tpep_pickup_datetime","5 minutes")
### 10. Calculer sur une fenêtre de 2 minutes / slide 1 minute :

In [28]:
#  Watermark + fenêtres glissantes
agg_df = parsed_df \
    .withWatermark("tpep_pickup_datetime", "5 minutes") \
    .groupBy(
        window("tpep_pickup_datetime", "2 minutes", "1 minute"),
        "PULocationID"
    ) \
    .agg(
        count("*").alias("nb_trips"),
        avg("fare_amount").alias("avg_fare")
    )


ConnectionRefusedError: [Errno 111] Connection refused

### 11. Afficher en console pour valider : writeStream.format("console").outputMode("update").start()

In [22]:
# # Affichage console pour valider
# query = agg_df.writeStream \
#     .format("console") \
#     .outputMode("update") \
#     .option("truncate", False) \
#     .start()

# query.awaitTermination()

## Écriture des résultats dans HDFSPrérequis

In [27]:
query = agg_df.writeStream \
    .format("parquet") \
    .option("path", "hdfs://namenode:8020/results/streaming/") \
    .option("checkpointLocation", "hdfs://namenode:8020/checkpoints/taxi/") \
    .outputMode("append") \
    .start()

query.awaitTermination()

ConnectionRefusedError: [Errno 111] Connection refused